In [2]:
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [3]:
data = pd.read_csv("Data/riceClassification.csv")
data.head()

,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,1,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,4,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,5,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


## Preprocessing 

In [15]:
data.dropna(inplace=True)
data.drop(['id'], axis=1, inplace=True)
data

KeyError: "['id'] not found in axis"

In [ ]:
print(data['Class'].unique())
print(data["Class"].value_counts())

[1 0]
Class
1    9985
0    8200
Name: count, dtype: int64


- Here we will normalize the data because the weights are smaller compared to the dataset values, the weights will be multiplied with the inputs, so they will get huge and slower.
- We will make each column to be the max value = 1.


In [ ]:
original_df = data.copy()

for column in data.columns:
    data[column] = data[column] / data[column].max() # Divide each value of each column with the max value of each column
data.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,0.444368,0.503404,0.775435,0.744658,0.424873,0.666610,0.741661,0.537029,0.844997,0.368316,1.0
1,0.281293,0.407681,0.622653,0.750489,0.273892,0.530370,0.804230,0.409661,0.919215,0.371471,1.0
2,0.298531,0.416421,0.630442,0.756341,0.284520,0.546380,0.856278,0.412994,0.959862,0.374747,1.0
3,0.300979,0.420463,0.629049,0.764024,0.286791,0.548616,0.883772,0.414262,0.961818,0.379222,1.0
4,0.361704,0.464626,0.682901,0.775033,0.345385,0.601418,0.867808,0.452954,0.966836,0.386007,1.0


In [ ]:

# : (Before the comma): This tells Python to keep all rows.
# : (Inside the slice): The colon indicates a range from the start to a specific stop point.
# -1 (Inside the slice): This represents the very last element or index. 

# Inputs
X = np.array(data.iloc[:, :-1]) # Take all the column from the first one unitl the end except the last column in the data

# Output
y = np.array(data.iloc[:, -1])


In [ ]:
# Spliting the data 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3)

In [ ]:
# We will split the testing data (which is 30%) into 50% testing and 50% validation 
# Validation set --> You use validation data while developing the model. You check the validation score and make improvements.
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5)
# Test set --> "How good is this model on data it has never seen?" / The test set should be untouched during development.


# Why not just use validation data?
# Because if you repeatedly check validation performance and change your model based on it, your model starts indirectly adapting to the validation set.

In [ ]:
print(X_train.shape)

(12729, 10)


- PyTorch is like a whole separate system, so we need to convert our pandas dataset object into PyTorch dataset object.
- Pandas stores / clean / manipulate the data, However, Dataset from PyTorch prepares data for training.
- When you train a neural network, PyTorch doesn't want a DataFrame. It wants data in a format that it can efficiently feed to the model which is Dataset.

In [ ]:
class MyDataset(Dataset):
    # Dataset class already has methods, we will override them and modify some of them to match our data properties

    def __init__(self, X, y): # Take the inputs and the output 
        self.X = torch.tensor(X, dtype=torch.float32).to(device) # to dvice --> GPU
        self.y = torch.tensor(y, data=torch.float32).to(device)

    # Override from the parent, so it matches our data
    def __len__(self):
        return len(self.X) # returns the total number of samples in the dataset

    # Override
    def __getitem__(self, index): # returns one sample given its index.
        return self.X[index], self.y[index]
